# Islander Recruitment

Selects 60 random islanders with the inclusion criteria 18 <= age <= 60, and requests consent from them.

In [1]:
from api.API import IslandsAPI # Import islands API
from api.person import Person #Import Person type
import pandas as pd # import pandas for dataframe manipulations
from random import choice # library for random sampling
from tqdm.notebook import tqdm

tqdm.pandas()

api = IslandsAPI() #initialize the api

Create a function to select a random islander.

In [6]:
houses = sum([village.get_houses() for village in api.get_villages()],[]) #Get every house and turn it into one large list

def get_random_person() -> Person:
    house = choice(houses) #Select a random house
    if house.is_empty():
        return get_random_person() #If the house is empty, rerun get_random Person
    residents = house.get_residents()
    if len(residents) == 0:
        return get_random_person()#If the house is empty, rerun get_random Person
    res : Person = choice(residents) #Select a random resident from the house
    return res

Enroll 60 islanders in the study.

In [14]:
def inclusion_criteria(person : Person) -> bool:
    return 18 <= person.get_age() <= 60

p_list = [] #initialize empty list
for i in tqdm(range(0,60), desc='Recruiting Islanders'): # Count 60 times
    while True:
        p = get_random_person() #Get a random person
        if p in p_list or not inclusion_criteria(p): #Make sure person isn't already in our list of participants
            continue
        if p.request_consent(): #Request Consent
            while not p.toggle_contact(): #Add person as a contact
                pass
            p_list.append(p) #Add person to list of contacts
            break #Go to next person

Save the list of islanders as a csv file

In [17]:
data = [(p.get_id(),p.get_name(),p.get_age()) for p in api.get_study_participants()] #Get all study participants into a list
data = pd.DataFrame(data, columns=["id","name","age"]) #Convert to a pandas dataframe
data['name'] = data['name'].str.replace('<br>',' ') # Replace "<br>"s with spaces
data.head() # Show the first 5 people

,id,name,age
0,bca8bah4ke,Husaam Agarwal,25
1,m4vtt3nf5y,Nirvan Babu,46
2,ftf4qwq8rl,Felix Bager,19
3,czwugg2hn6,Yorick Blomgren,25
4,wj9trvcqc9,Rhona Bruce,47


In [14]:
data.to_csv('islanders.csv') # Save as a csv